In [4]:

import os
from uuid import uuid4
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()


# 1. LLM
model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="low"
)


# 2. Structured output for human decision
class HumanDecision(BaseModel):
    decision: Literal["refine", "send"] = Field(
        description="Whether the user wants to improve the email or approve it"
    )
    feedback: str = Field(
        description="The requested email changes, or an empty string if approved"
    )


decision_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are interpreting the user's response to an email draft.

    Decide what the user wants:

    - refine: The user requests changes, improvements, or corrections.
    - send: The user is satisfied and explicitly wants to send or finalize it.

    Extract the requested changes into feedback.

    If the user's intention is unclear, choose refine and ask for clarification
    in the feedback. Never assume approval.
    """),
    ("human", """
    Current email draft:
    {draft}

    User response:
    {user_message}
    """)
])

decision_chain = decision_prompt | model.with_structured_output(HumanDecision,method="function_calling")


# 3. Graph state
class AgentState(BaseModel):
    request: str
    draft: str = ""
    user_message: str = ""
    feedback: str = ""
    decision: Literal["refine", "send"] | None = None


# 4. Draft node: create or improve the email
def draft_node(state: AgentState):

    if state.draft and state.feedback:
        prompt = f"""
        Improve the following email based on the user's feedback.

        Current email:
        {state.draft}

        User feedback:
        {state.feedback}

        Return the complete improved email only.
        """
    else:
        prompt = f"""
        Write a professional email based on this request:

        {state.request}

        Return the complete email only.
        """

    response = model.invoke(prompt)

    return {
        "draft": response.content,
        "feedback": "",
        "user_message": "",
        "decision": None
    }


# 5. Human review: pause and collect natural language input
def review_node(state: AgentState):

    user_message = interrupt({
        "message": "Here is your email draft. What would you like to change?",
        "draft": state.draft
    })

    return {
        "user_message": user_message
    }


# 6. LLM interprets the human response
def decision_node(state: AgentState):

    result = decision_chain.invoke({
        "draft": state.draft,
        "user_message": state.user_message
    })

    print("\nLLM decision:", result)

    return {
        "decision": result.decision,
        "feedback": result.feedback
    }


# 7. Conditional routing
def route_decision(state: AgentState):
    return state.decision


# 8. Build graph
builder = StateGraph(AgentState)

builder.add_node("draft", draft_node)
builder.add_node("review", review_node)
builder.add_node("decision", decision_node)

builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", "decision")

builder.add_conditional_edges(
    "decision",
    route_decision,
    {
        "refine": "draft",
        "send": END
    }
)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)




In [ ]:
while True:

    request = input("\nWhat email would you like to write? ")

    if request.lower() in ["exit", "quit"]:
        break

    config = {
        "configurable": {
            "thread_id": str(uuid4())
        }
    }

    result = graph.invoke(
        {"request": request},
        config=config
    )

    # Continue interacting while the graph is interrupted
    while result.get("__interrupt__"):

        payload = result["__interrupt__"][0].value

        print("\n========== EMAIL DRAFT ==========")
        print(payload["draft"])
        print("=================================")
        print(payload["message"])

        user_message = input("\nYour response: ")

        result = graph.invoke(
            Command(resume=user_message),
            config=config
        )

    print("\n========== FINAL EMAIL ==========")
    print(result["draft"])
    print("Email approved!")
    print("=================================")
#interrupt gives the payload and resume send data to interrupt so when we resume the graph go back to interump with the resume value


========== EMAIL DRAFT ==========
Subject: Project Timeline Update

Dear [Manager's Name],

I hope you are well.

I am writing to provide an update on the current project timeline and to flag a few items that may impact our delivery dates.

As of today, we are on track with Phase 1 and Phase 2 deliverables, which remain scheduled for completion by [Date]. However, we have identified a potential delay in the design review stage due to pending feedback from the client. If the feedback is not received by [Date], we anticipate a 3-4 day shift in the subsequent development milestone.

To mitigate this risk, I recommend we:
1. Schedule a checkpoint meeting with the client by [Date] to confirm review timelines
2. Reallocate resources to parallel workstreams where possible

Please let me know if you would like a detailed revised timeline or if you would prefer to discuss this in our next 1:1.

Thank you for your guidance.

Best regards,
[Your Name]
[Your Position]
[Contact Information]
Here i